# STAGE 1 - IMPORTS AND CONFIGURATION

In [6]:
import os
import time
import asyncio
import logging
import numpy as np
import pandas as pd
import nltk
import instructor

from dotenv import load_dotenv
from openai import AsyncOpenAI

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate

from bert_score import BERTScorer
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score

from ragas.embeddings import OpenAIEmbeddings as RagasEmbeddings
from ragas.llms import InstructorLLM
from ragas.metrics.collections import AnswerCorrectness

from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import (
    GEval,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    FaithfulnessMetric
)

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env")

logging.getLogger("deepeval").setLevel(logging.CRITICAL)
logging.getLogger("transformers").setLevel(logging.ERROR)

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

INPUT_CSV = "ai_teacher_20_questions.csv"
PDF_FILE = "HopeAI_Teaching.pdf"
OUTPUT_CSV = "RAG_LLM_FINAL_EVALUATION.csv"

MODEL = "gpt-4o-mini"
EVALUATION_MODEL = "gpt-4.1"

# Configure these according to the current API pricing.
INPUT_PRICE = 0.15 / 1_000_000
OUTPUT_PRICE = 0.60 / 1_000_000

df = pd.read_csv(INPUT_CSV)

required_columns = ["User Query", "Teacher Response"]
missing = set(required_columns) - set(df.columns)

if missing:
    raise ValueError(f"Missing CSV columns: {missing}")

df = df.dropna(subset=required_columns).reset_index(drop=True)

df["User Query"] = df["User Query"].astype(str).str.strip()
df["Teacher Response"] = df["Teacher Response"].astype(str).str.strip()

print(f"Questions loaded: {len(df)}")

Questions loaded: 20


C:\Users\User\AppData\Local\Temp\ipykernel_35500\1418208404.py:27: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


# STAGE 2 - RAG PIPELINE

In [7]:
# LOAD PDF
documents = PyPDFLoader(PDF_FILE).load()

# CHUNK DOCUMENTS
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = text_splitter.split_documents(documents)

# CREATE FAISS VECTOR STORE
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# LLM
gpt_model = ChatOpenAI(model=MODEL, temperature=0, timeout=120, max_retries=3)

# RAG PROMPT

rag_prompt = ChatPromptTemplate.from_template("""
Answer the question using only the retrieved context.

If the answer is not available in the context,
say that the information could not be found in the documents.

Context:
{context}

Question:
{question}

Answer:
""")

# RETRIEVE CONTEXT

def get_context(query):
    retrieved_docs = retriever.invoke(query)
    retrieval_context = [
        doc.page_content for doc in retrieved_docs
    ]
    context = "\n\n".join(retrieval_context)
    return context, retrieval_context

print("PDF pages loaded:", len(documents))
print("Chunks created:", len(chunks))
print("FAISS vector store and top-3 retriever ready.")

PDF pages loaded: 6
Chunks created: 18
FAISS vector store and top-3 retriever ready.


# STAGE 3 - DEFINE EVALUATION METRICS

In [8]:
# SIMILARITY METRICS

bert_scorer = BERTScorer(lang="en", rescale_with_baseline=False)
rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

# ANSWER QUALITY METRICS
correctness_metric = GEval(
    name="Answer Correctness",
    model=EVALUATION_MODEL,
    criteria="""
    Compare the generated answer with the reference answer.
    Evaluate factual accuracy, important information,
    and contradictions. Ignore minor wording differences.
    """,
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ],
    async_mode=False
)

helpfulness_metric = GEval(
    name="Helpfulness",
    model=EVALUATION_MODEL,
    criteria="""
    Evaluate whether the answer directly addresses the question,
    is easy for a beginner to understand, and gives a clear,
    useful explanation without unnecessary information.
    """,
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    async_mode=False
)
answer_relevancy_metric = AnswerRelevancyMetric(model=EVALUATION_MODEL, include_reason=False, async_mode=False)

# RAGAS ANSWER CORRECTNESS
ragas_client = AsyncOpenAI(api_key=api_key)
instructor_client = instructor.from_openai(ragas_client)
ragas_llm = InstructorLLM(
    client=instructor_client,
    model="gpt-5-nano",
    provider="openai",
    max_tokens=8192,
    temperature=1.0
)
ragas_embedding_client = AsyncOpenAI(api_key=api_key)
ragas_embedding_model = RagasEmbeddings(client=ragas_embedding_client, model="text-embedding-3-small")
answer_correctness_metric = AnswerCorrectness(llm=ragas_llm, embeddings=ragas_embedding_model)

# RAG GEVAL
rag_geval_metric = GEval(
    name="RAG Correctness",
    model="gpt-5-nano",
    criteria="""
    Evaluate whether the generated answer is factually correct
    and consistent with the reference answer.
    """,
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ],
    async_mode=False
)

# RETRIEVAL / RAG METRICS
context_precision_metric = ContextualPrecisionMetric(model=EVALUATION_MODEL)
context_recall_metric = ContextualRecallMetric(model=EVALUATION_MODEL)
faithfulness_metric = FaithfulnessMetric(model=EVALUATION_MODEL)
rag_answer_relevancy_metric = AnswerRelevancyMetric(model=EVALUATION_MODEL, include_reason=False, async_mode=False)
print("All evaluation metrics initialized.")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

All evaluation metrics initialized.


# STAGE 4 - INPUT QUERIES, GENERATE ANSWERS AND CALCULATE OPERATIONAL METRICS

In [23]:
from tqdm.auto import tqdm

results = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="RAG generation"
):
    query = row["User Query"]
    reference = row["Teacher Response"]

    try:
        total_start = time.perf_counter()

        # RETRIEVAL
        start = time.perf_counter()
        context, retrieval_context = get_context(query)
        retrieval_latency = time.perf_counter() - start

        # GENERATION
        prompt_value = rag_prompt.invoke({
            "context": context,
            "question": query
        })

        start = time.perf_counter()
        response = gpt_model.invoke(prompt_value)
        generation_latency = time.perf_counter() - start

        total_latency = time.perf_counter() - total_start

        generated = response.content.strip()

        # TOKEN USAGE
        metadata = getattr(response, "response_metadata", {})
        usage = metadata.get("token_usage", {})

        input_tokens = usage.get("prompt_tokens", 0)
        output_tokens = usage.get("completion_tokens", 0)
        total_tokens = usage.get(
            "total_tokens",
            input_tokens + output_tokens
        )

        # COST
        input_cost = input_tokens * INPUT_PRICE
        output_cost = output_tokens * OUTPUT_PRICE
        total_cost = input_cost + output_cost

        results.append({
            "User Query": query,
            "Teacher Response": reference,
            "GeneratedAnswer": generated,
            "RetrievedContext": retrieval_context,

            "RetrievalLatency": retrieval_latency,
            "GenerationLatency": generation_latency,
            "Latency": total_latency,

            "InputTokens": input_tokens,
            "OutputTokens": output_tokens,
            "TotalTokens": total_tokens,

            "InputCost": input_cost,
            "OutputCost": output_cost,
            "TotalCost": total_cost
        })

    except Exception as e:
        print(f"\nError: {query}\n{e}")

        results.append({
            "User Query": query,
            "Teacher Response": reference,
            "GeneratedAnswer": "",
            "RetrievedContext": [],

            "RetrievalLatency": np.nan,
            "GenerationLatency": np.nan,
            "Latency": np.nan,

            "InputTokens": np.nan,
            "OutputTokens": np.nan,
            "TotalTokens": np.nan,

            "InputCost": np.nan,
            "OutputCost": np.nan,
            "TotalCost": np.nan
        })


results_df = pd.DataFrame(results)

display(results_df)

ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-2784' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\User\anaconda3\envs\ai\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-2785' coro=<Kernel.shell_main() running at C:\Users\User\anaconda3\envs\ai\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\User\anaconda3\envs\ai\Lib\site-packages\zmq\eventloop\zmqstream.py:564]>
Exception ignored in: <coroutine object Kernel.shell_main at 0x000002158B32F740>
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'
Exception ignored in: <coroutine object Kernel.shell_main at 0x000002158B32F740>
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'
ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-2785' coro=<Kernel

RAG generation:   0%|          | 0/20 [00:00<?, ?it/s]

,User Query,Teacher Response,GeneratedAnswer,RetrievedContext,RetrievalLatency,GenerationLatency,Latency,InputTokens,OutputTokens,TotalTokens,InputCost,OutputCost,TotalCost
0,What is AI?,AI (Artificial Intelligence) is the ability of...,"AI, or artificial intelligence, is the ability...",[when actions are ambiguous or poorly controll...,2.897937,2.375597,5.314380,534,72,606,0.000080,0.000043,0.000123
1,How does AI differ from traditional programming?,Traditional programs follow fixed rules writte...,AI differs from traditional programming mainly...,[knowledge for a small educational chatbot or ...,0.901457,1.767215,2.671938,533,87,620,0.000080,0.000052,0.000132
2,What is Machine Learning?,Machine Learning is a branch of AI where compu...,Machine learning is a branch of artificial int...,[The relationship among the three concepts is ...,0.278740,1.527383,1.809496,394,58,452,0.000059,0.000035,0.000094
3,What is Deep Learning?,Deep Learning is a type of Machine Learning th...,Deep learning is a specialized area within mac...,[The relationship among the three concepts is ...,0.254192,1.455021,1.711368,447,47,494,0.000067,0.000028,0.000095
4,"What is the difference between AI, ML, and Dee...",AI is the broad field of making machines intel...,"The difference among AI, ML, and Deep Learning...",[The relationship among the three concepts is ...,0.261940,2.178629,2.443208,455,116,571,0.000068,0.000070,0.000138
5,What is training data?,Training data is the data used to teach a Mach...,Training data is the collection of examples us...,[AI TEACHING DATASET - PARAGRAPH DOCUMENTATION...,3.787196,1.888601,5.679920,489,72,561,0.000073,0.000043,0.000117
6,What is a Machine Learning model?,A Machine Learning model is a mathematical sys...,A machine learning model is the mathematical s...,"[recommendation, or decision. The model does n...",0.302687,1.308059,1.615325,461,67,528,0.000069,0.000040,0.000109
7,What is supervised learning?,Supervised learning trains a model using examp...,Supervised learning uses examples that contain...,"[recommendation, or decision. The model does n...",0.242720,1.562164,1.807614,406,64,470,0.000061,0.000038,0.000099
8,What is unsupervised learning?,Unsupervised learning works with data that doe...,Unsupervised learning works with information t...,"[recommendation, or decision. The model does n...",0.255543,1.258832,1.515654,408,70,478,0.000061,0.000042,0.000103
9,What is reinforcement learning?,Reinforcement learning teaches an agent throug...,Reinforcement learning teaches an agent throug...,"[model searches for hidden structures, similar...",0.251749,1.313140,1.566590,531,65,596,0.000080,0.000039,0.000119


# STAGE 5 - LLM EVALUATION - ANSWER QUALITY

In [20]:
quality_scores = []

for _, row in tqdm(
    results_df.iterrows(),
    total=len(results_df),
    desc="Answer quality"
):
    query = row["User Query"]
    generated = row["GeneratedAnswer"]
    reference = row["Teacher Response"]

    if not generated:
        quality_scores.append([np.nan] * 3)
        continue

    test_case = LLMTestCase(
        input=query,
        actual_output=generated,
        expected_output=reference
    )

    scores = []

    for metric in [
        correctness_metric,
        answer_relevancy_metric,
        helpfulness_metric
    ]:
        try:
            metric.measure(test_case)
            scores.append(float(metric.score))
        except Exception as e:
            print(f"{metric.__class__.__name__} error: {e}")
            scores.append(np.nan)

    quality_scores.append(scores)


results_df[
    ["AnswerCorrectness", "AnswerRelevancy", "Helpfulness"]
] = pd.DataFrame(
    quality_scores,
    index=results_df.index
)

results_df["AnswerQuality"] = results_df[
    ["AnswerCorrectness", "AnswerRelevancy", "Helpfulness"]
].mean(axis=1)

display(results_df[
    [
        "User Query",
        "AnswerCorrectness",
        "AnswerRelevancy",
        "Helpfulness",
        "AnswerQuality"
    ]
])

Answer quality:   0%|          | 0/20 [00:00<?, ?it/s]

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

,User Query,AnswerCorrectness,AnswerRelevancy,Helpfulness,AnswerQuality
0,What is AI?,0.900000,1.000000,1.000000,0.966667
1,How does AI differ from traditional programming?,0.911280,1.000000,1.000000,0.970427
2,What is Machine Learning?,0.920689,1.000000,1.000000,0.973563
3,What is Deep Learning?,0.912252,1.000000,0.908756,0.940336
4,"What is the difference between AI, ML, and Dee...",1.000000,1.000000,0.902596,0.967532
5,What is training data?,0.803210,1.000000,0.980317,0.927842
6,What is a Machine Learning model?,0.904209,1.000000,1.000000,0.968070
7,What is supervised learning?,0.993245,1.000000,1.000000,0.997748
8,What is unsupervised learning?,0.980317,1.000000,0.992630,0.990983
9,What is reinforcement learning?,0.908270,1.000000,1.000000,0.969423


# STAGE 6 - EVALUATE SIMILARITY METRICS

In [18]:
similarity_scores = []

for _, row in tqdm(
    results_df.iterrows(),
    total=len(results_df),
    desc="Similarity metrics"
):
    reference = row["Teacher Response"]
    generated = row["GeneratedAnswer"]

    if not generated:
        similarity_scores.append([np.nan] * 5)
        continue

    try:
        _, _, f1 = bert_scorer.score(
            [generated],
            [reference]
        )
        bert = float(f1[0])
    except Exception:
        bert = np.nan

    try:
        rouge_scores = rouge.score(reference, generated)

        rouge1 = rouge_scores["rouge1"].fmeasure
        rouge2 = rouge_scores["rouge2"].fmeasure
        rougeL = rouge_scores["rougeL"].fmeasure
    except Exception:
        rouge1 = rouge2 = rougeL = np.nan

    try:
        meteor = meteor_score(
            [reference.split()],
            generated.split()
        )
    except Exception:
        meteor = np.nan

    similarity_scores.append([
        bert, rouge1, rouge2, rougeL, meteor
    ])


similarity_columns = [
    "BERT",
    "ROUGE-1",
    "ROUGE-2",
    "ROUGE-L",
    "METEOR"
]

results_df[similarity_columns] = pd.DataFrame(
    similarity_scores,
    index=results_df.index
)

display(results_df[["User Query"] + similarity_columns])

Similarity metrics:   0%|          | 0/20 [00:00<?, ?it/s]

,User Query,BERT,ROUGE-1,ROUGE-2,ROUGE-L,METEOR
0,What is AI?,0.936465,0.541176,0.385542,0.494118,0.697060
1,How does AI differ from traditional programming?,0.896392,0.326923,0.196078,0.288462,0.466625
2,What is Machine Learning?,0.909317,0.444444,0.278481,0.444444,0.457683
3,What is Deep Learning?,0.941287,0.606061,0.468750,0.606061,0.739874
4,"What is the difference between AI, ML, and Dee...",0.913300,0.597701,0.352941,0.528736,0.558382
5,What is training data?,0.895691,0.391304,0.200000,0.347826,0.465113
6,What is a Machine Learning model?,0.900434,0.450000,0.153846,0.300000,0.442387
7,What is supervised learning?,0.913420,0.457831,0.172840,0.337349,0.484613
8,What is unsupervised learning?,0.928887,0.439024,0.175000,0.365854,0.503774
9,What is reinforcement learning?,0.926493,0.453333,0.219178,0.346667,0.616099


# STAGE 7 - RAG EVALUATION METRICS

In [19]:
async def calculate_ragas(query, reference, generated):
    try:
        score = await answer_correctness_metric.ascore(
            user_input=query,
            response=generated,
            reference=reference
        )
        return float(score.value)
    except Exception as e:
        print("RAGAS error:", e)
        return np.nan


rag_scores = []

for _, row in tqdm(
    results_df.iterrows(),
    total=len(results_df),
    desc="RAG evaluation"
):
    query = row["User Query"]
    reference = row["Teacher Response"]
    generated = row["GeneratedAnswer"]
    context = row["RetrievedContext"]

    if not generated:
        rag_scores.append([np.nan] * 6)
        continue

    # RAGAS
    ragas_score = await calculate_ragas(
        query, reference, generated
    )

    # DEEPEVAL RAG METRICS
    test_case = LLMTestCase(
        input=query,
        actual_output=generated,
        expected_output=reference,
        retrieval_context=context
    )

    metric_values = []

    for metric in [
        context_precision_metric,
        context_recall_metric,
        faithfulness_metric,
        rag_answer_relevancy_metric,
        rag_geval_metric
    ]:
        try:
            metric.measure(test_case)
            metric_values.append(float(metric.score))
        except Exception as e:
            print(f"{metric.__class__.__name__} error: {e}")
            metric_values.append(np.nan)

    # Order: RAGAS, precision, recall, faithfulness,
    # answer relevancy, GEval
    rag_scores.append([
        ragas_score,
        *metric_values
    ])


rag_columns = [
    "Ragas AnswerCorrectness",
    "Context Precision",
    "Context Recall",
    "Faithfulness",
    "RAG Answer Relevancy",
    "RAG GEval"
]

results_df[rag_columns] = pd.DataFrame(
    rag_scores,
    index=results_df.index
)

results_df["RAG Evaluation Score"] = results_df[
    rag_columns
].mean(axis=1)

display(results_df[["User Query"] + rag_columns + [
    "RAG Evaluation Score"
]])

RAG evaluation:   0%|          | 0/20 [00:00<?, ?it/s]

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

,User Query,Ragas AnswerCorrectness,Context Precision,Context Recall,Faithfulness,RAG Answer Relevancy,RAG GEval,RAG Evaluation Score
0,What is AI?,0.979948,0.5,1.0,1.0,1.0,0.8,0.879991
1,How does AI differ from traditional programming?,0.928222,1.0,1.0,1.0,1.0,0.9,0.971370
2,What is Machine Learning?,0.678200,1.0,1.0,1.0,1.0,0.7,0.896367
3,What is Deep Learning?,0.796950,1.0,1.0,1.0,1.0,0.7,0.916158
4,"What is the difference between AI, ML, and Dee...",0.948836,1.0,1.0,1.0,1.0,0.9,0.974806
5,What is training data?,0.554960,1.0,1.0,1.0,1.0,0.7,0.875827
6,What is a Machine Learning model?,0.548598,1.0,1.0,1.0,1.0,1.0,0.924766
7,What is supervised learning?,0.647569,1.0,1.0,1.0,1.0,0.9,0.924595
8,What is unsupervised learning?,0.715114,1.0,1.0,1.0,1.0,0.7,0.902519
9,What is reinforcement learning?,0.547397,1.0,1.0,1.0,1.0,0.4,0.824566


# STAGE 8 - OPERATIONAL METRICS SUMMARY

In [21]:
operational_columns = [
    "RetrievalLatency",
    "GenerationLatency",
    "Latency",
    "InputTokens",
    "OutputTokens",
    "TotalTokens",
    "InputCost",
    "OutputCost",
    "TotalCost"
]

operational_summary = pd.DataFrame({
    "Metric": operational_columns,
    "Average": [
        results_df[c].mean() for c in operational_columns
    ],
    "Total": [
        results_df[c].sum() for c in operational_columns
    ]
})

display(operational_summary.round(6))

,Metric,Average,Total
0,RetrievalLatency,0.398306,7.966110
1,GenerationLatency,1.590429,31.808588
2,Latency,1.991659,39.833170
3,InputTokens,472.300000,9446.000000
4,OutputTokens,71.250000,1425.000000
5,TotalTokens,543.550000,10871.000000
6,InputCost,0.000071,0.001417
7,OutputCost,0.000043,0.000855
8,TotalCost,0.000114,0.002272


# STAGE 9 - FINAL EVALUATION TABLE SUMMARY

In [22]:
# ROUND ALL METRIC COLUMNS

metric_columns = [
    "AnswerCorrectness",
    "AnswerRelevancy",
    "Helpfulness",
    "AnswerQuality",
    *similarity_columns,
    *rag_columns,
    "RAG Evaluation Score"
]

results_df[metric_columns] = results_df[metric_columns].round(3)


# FINAL QUESTION-WISE TABLE

final_columns = [
    "User Query",
    "Teacher Response",
    "GeneratedAnswer",
    *metric_columns,
    *operational_columns
]

final_df = results_df[final_columns].copy()

display(final_df)


# AVERAGE EVALUATION SUMMARY

summary_df = pd.DataFrame({
    "Metric": metric_columns,
    "Average Score": [
        results_df[c].mean() for c in metric_columns
    ]
})

summary_df["Average Score"] = (
    summary_df["Average Score"].round(3)
)

display(summary_df)


# SAVE OUTPUTS

final_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    "RAG_LLM_EVALUATION_SUMMARY.csv",
    index=False,
    encoding="utf-8-sig"
)

operational_summary.to_csv(
    "RAG_LLM_OPERATIONAL_SUMMARY.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Evaluation completed.")
print("Final results:", OUTPUT_CSV)
print("Evaluation summary: RAG_LLM_EVALUATION_SUMMARY.csv")
print("Operational summary: RAG_LLM_OPERATIONAL_SUMMARY.csv")

,User Query,Teacher Response,GeneratedAnswer,AnswerCorrectness,AnswerRelevancy,Helpfulness,AnswerQuality,BERT,ROUGE-1,ROUGE-2,...,RAG Evaluation Score,RetrievalLatency,GenerationLatency,Latency,InputTokens,OutputTokens,TotalTokens,InputCost,OutputCost,TotalCost
0,What is AI?,AI (Artificial Intelligence) is the ability of...,"AI, or artificial intelligence, is the ability...",0.900,1.000,1.000,0.967,0.936,0.541,0.386,...,0.880,1.441383,3.536660,4.990236,534,72,606,0.000080,0.000043,0.000123
1,How does AI differ from traditional programming?,Traditional programs follow fixed rules writte...,AI differs from traditional programming mainly...,0.911,1.000,1.000,0.970,0.896,0.327,0.196,...,0.971,0.718855,1.794741,2.514886,533,87,620,0.000080,0.000052,0.000132
2,What is Machine Learning?,Machine Learning is a branch of AI where compu...,Machine learning is a branch of artificial int...,0.921,1.000,1.000,0.974,0.909,0.444,0.278,...,0.896,0.257677,1.422330,1.681944,394,58,452,0.000059,0.000035,0.000094
3,What is Deep Learning?,Deep Learning is a type of Machine Learning th...,Deep learning is a specialized area within mac...,0.912,1.000,0.909,0.940,0.941,0.606,0.469,...,0.916,0.241056,1.090079,1.332338,447,47,494,0.000067,0.000028,0.000095
4,"What is the difference between AI, ML, and Dee...",AI is the broad field of making machines intel...,"The difference among AI, ML, and Deep Learning...",1.000,1.000,0.903,0.968,0.913,0.598,0.353,...,0.975,0.382961,1.331282,1.716430,455,64,519,0.000068,0.000038,0.000107
5,What is training data?,Training data is the data used to teach a Mach...,Training data is the collection of examples us...,0.803,1.000,0.980,0.928,0.896,0.391,0.200,...,0.876,0.300669,1.430209,1.733045,489,72,561,0.000073,0.000043,0.000117
6,What is a Machine Learning model?,A Machine Learning model is a mathematical sys...,A machine learning model is the mathematical s...,0.904,1.000,1.000,0.968,0.900,0.450,0.154,...,0.925,0.306214,1.535233,1.843204,461,67,528,0.000069,0.000040,0.000109
7,What is supervised learning?,Supervised learning trains a model using examp...,Supervised learning uses examples that contain...,0.993,1.000,1.000,0.998,0.913,0.458,0.173,...,0.925,0.303528,1.226223,1.532739,406,64,470,0.000061,0.000038,0.000099
8,What is unsupervised learning?,Unsupervised learning works with data that doe...,Unsupervised learning works with information t...,0.980,1.000,0.993,0.991,0.929,0.439,0.175,...,0.903,0.306153,1.324377,1.632719,408,70,478,0.000061,0.000042,0.000103
9,What is reinforcement learning?,Reinforcement learning teaches an agent throug...,Reinforcement learning teaches an agent throug...,0.908,1.000,1.000,0.969,0.926,0.453,0.219,...,0.825,0.305573,1.634641,1.942518,531,66,597,0.000080,0.000040,0.000119


,Metric,Average Score
0,AnswerCorrectness,0.907
1,AnswerRelevancy,0.983
2,Helpfulness,0.974
3,AnswerQuality,0.955
4,BERT,0.912
5,ROUGE-1,0.430
6,ROUGE-2,0.239
7,ROUGE-L,0.368
8,METEOR,0.501
9,Ragas AnswerCorrectness,0.691


Evaluation completed.
Final results: RAG_LLM_FINAL_EVALUATION.csv
Evaluation summary: RAG_LLM_EVALUATION_SUMMARY.csv
Operational summary: RAG_LLM_OPERATIONAL_SUMMARY.csv
